#### 벡터 데이터베이스와 임베딩 모델 성능 비교

In [ ]:
'''
RAG의 과정에서 R에 해당하는 유사항목 검색을 위해서
데이터를 벡터화하여 저장하는 벡터 데이터 베이스를 이용한다.


# 저장되는 구조
{
    "vector": [0.12, -0.34, 0.56, ...],   # 임베딩 벡터 (검색용)
    "content": "복구충당부채란 미래에...",   # 원본 텍스트
    "metadata": {                          # 부가 정보
        "source": "회계기준서.pdf",
        "page": 42,
        "chapter": "충당부채",
        "date": "2024-01-15"
    }
}

임베딩 모델을 이용해 데이터를 이러한 구조로 저장한 것을 벡터 데이터베이스라고 한다.
'''

In [ ]:
'''
이러한 벡터 데이터베이스가 마련되었다면

RAG 과정 가운데 

Retrieval (유사도 검색)
Augmented ()
'''

In [1]:
%pip install -q python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [7]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
%pip install google-genai


   ---------------------------------------- 0/5 [websockets]
   -------- ------------------------------- 1/5 [rsa]
  Attempting uninstall: anyio
   -------- ------------------------------- 1/5 [rsa]
    Found existing installation: anyio 4.7.0
   -------- ------------------------------- 1/5 [rsa]
    Uninstalling anyio-4.7.0:
   -------- ------------------------------- 1/5 [rsa]
      Successfully uninstalled anyio-4.7.0
   -------- ------------------------------- 1/5 [rsa]
   ---------------- ----------------------- 2/5 [anyio]
   ---------------- ----------------------- 2/5 [anyio]
   ------------------------ --------------- 3/5 [google-auth]
   ------------------------ --------------- 3/5 [google-auth]
   -------------------------------- ------- 4/5 [google-genai]
   -------------------------------- ------- 4/5 [google-genai]
   -------------------------------- ------- 4/5 [google-genai]
   -------------------------------- ------- 4/5 [google-genai]
   -----------------------------

In [8]:
from openai import OpenAI
client = OpenAI()

#### 벡터 유사도 산출 함수

In [9]:
import numpy as np


## 벡터 유사도
def cosine_similarity(vec1, vec2):
  '''
  두 벡터 사이의 코사인 유사도 계산
  -1 ~ 1 사이의 값이 도출되며,
  데이터베이스 내에서 1에 가까운 값들 k개를 기준으로 답변을 생성하게 됨
  '''
  dot_product = np.dot(vec1, vec2)
  norm_vec1 = np.linalg.norm(vec1)
  norm_vec2 = np.linalg.norm(vec2)
  
  if norm_vec1 == 0 or norm_vec2 == 0:
    return 0.0
  
  return dot_product / (norm_vec1 * norm_vec2)

#### openai로 임베딩

In [ ]:

king_embedding_response = client.embeddings.create(
  input= 'king',
  model= 'text-embedding-3-large'
)

In [13]:
king_embedding_response

'''
CreateEmbeddingResponse(
    data=[
        Embedding(
            embedding=[0.0104, 0.0249, -0.0014, ...],  # 실제 벡터 (3072차원 ㄷㄷㄷ)
            index=0,
            object='embedding'
        )
    ],
    model='text-embedding-3-large',
    object='list',
    usage=Usage(prompt_tokens=1, total_tokens=1)
)
'''

## Embedding() 안에 embedding 리스트가 있는 이유는
## 여러 단어를 임베딩하는 배치 처리를 위해서(나중에 봄)

CreateEmbeddingResponse(data=[Embedding(embedding=[0.01040416955947876, 0.024995191022753716, -0.0014775966992601752, 0.0033329545985907316, 0.0006571469129994512, 0.02008429728448391, 0.013937810435891151, 0.010349079966545105, -0.013662359677255154, 0.04429248720407486, 0.002467252081260085, -0.004214397165924311, -0.059245530515909195, -0.056569721549749374, 0.0314643494784832, 0.006961035076528788, -0.06522674858570099, -0.004119956865906715, -0.05521607771515846, -0.00014768588880542666, 0.01193095464259386, -0.05143846943974495, -0.0010417941957712173, 0.004533133003860712, 0.029449624940752983, -0.019438955932855606, -0.007405691314488649, 0.0033604996278882027, -0.015244233421981335, 0.008609804324805737, 0.006846919655799866, 0.05952885001897812, 0.013882719911634922, 0.038311269134283066, -0.011852254159748554, 0.0010801606113091111, 0.01693628914654255, -0.00979817844927311, -0.011364312842488289, -0.02227216400206089, -0.015771524980664253, -0.04136483743786812, -0.06862659

In [ ]:
king_vector = np.array(king_embedding_response.data[0].embedding)

# 위 CreateEmbeddingResponse 객체 속
# data[0] - Embedding()
# 중에서 embedding 리스트를
# np.array로 행렬로 만들어 king_vector에 할당함

In [12]:
king_vector

array([ 0.01040417,  0.02499519, -0.0014776 , ...,  0.00835009,
        0.01049861, -0.00254005])

In [16]:
# queen이라는 단어도 임베딩해 벡터를 산출하고
queen_embedding_response = client.embeddings.create(
  input= 'queen',
  model= 'text-embedding-3-large'
)

queen_vector = np.array(queen_embedding_response.data[0].embedding)

In [17]:
king_queen_simuarity = cosine_similarity(king_vector, queen_vector)

print(king_queen_simuarity)

0.5551271800059485


In [18]:
# 비교를 위해 slave 단어도 임베딩
slave_embedding_response = client.embeddings.create(
  input= 'slave',
  model= 'text-embedding-3-large'
)

slave_vector = np.array(slave_embedding_response.data[0].embedding)

In [20]:
king_slave_simuarity = cosine_similarity(king_vector, slave_vector)

print(king_slave_simuarity)

## king은 slave 보다는 queen에 더 가까운 단어임

0.29478135840993636


#### 한국어 embedding에 더 적합하다는 upstage 임베딩!

In [4]:
%pip install openai

Note: you may need to restart the kernel to use updated packages.


In [ ]:
from openai import OpenAI # openai==1.52.2
 
upstage_client = OpenAI(
  api_key="up_s2DscEn9zKrNMJMMqpCTpVwKKuvNk",
  base_url="https://api.upstage.ai/v1"
)
 
# 한국어 '왕' 임베딩
kr_king_embedding_response = upstage_client.embeddings.create(
  input= '왕',
  model="embedding-query"
)

kr_king_vector = np.array(kr_king_embedding_response.data[0].embedding)


# upstage로 king 임베딩
en_king_embedding_response = upstage_client.embeddings.create(
  input= 'king',
  model="embedding-query"
)

en_king_vector = np.array(en_king_embedding_response.data[0].embedding)

In [26]:
## upstage에서 임베딩한 king과 왕의 벡터 유사도

upstage_king_wang_simuarity = cosine_similarity(kr_king_vector, en_king_vector)

print(upstage_king_wang_simuarity)

0.852149171074866


In [27]:
## 반면에.. OpenAI king과 왕의 벡터 유사도는?
wang_embedding_response = client.embeddings.create(
  input= '왕',
  model= 'text-embedding-3-large'
)

wang_vector = np.array(wang_embedding_response.data[0].embedding)


openai_king_wang_simuarity = cosine_similarity(king_vector, wang_vector)

print(openai_king_wang_simuarity)

## openAI는 한글 임베딩에 보다 적절하지 않을 수 있다 - 한국어 임베딩에는 upstage(?)
## 근데 그러면 한국어 로컬 임베딩 모델은??

0.5474873912140231
